# Setup

Imports, device, and utils


In [1]:
from IPython import get_ipython

ip = get_ipython()
if not ip.extension_manager.loaded:
    ip.extension_manager.load("autoreload")
    %autoreload 2

In [2]:
import plotly.io as pio

pio.renderers.default = "notebook_connected"

In [3]:
import circuitsvis as cv

# Testing that the library works
cv.examples.hello("Andrew")

In [4]:
# Import stuff
import torch
import torch.nn as nn
import einops
import numpy as np
from circuitsvis.attention import attention_heads
from fancy_einsum import einsum
from IPython.display import HTML, IFrame
import tqdm.auto as tqdm
import plotly.express as px

from jaxtyping import Float
from functools import partial
from transformers import AutoModelForCausalLM, AutoTokenizer

In [5]:
# import transformer_lens
import transformer_lens.utils as utils
from transformer_lens.hook_points import (
    HookPoint,
)  # Hooking utilities
from transformer_lens import ActivationCache, HookedTransformer, FactoredMatrix

In [6]:
# save GPU mem for inference
torch.set_grad_enabled(False)
print("Disabled automatic differentiation")

# clear cache
torch.cuda.empty_cache()

Disabled automatic differentiation


In [7]:
# plotting helpers
def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.imshow(
        utils.to_numpy(tensor),
        color_continuous_midpoint=0.0,
        color_continuous_scale="RdBu",
        labels={"x": xaxis, "y": yaxis},
        **kwargs,
    ).show(renderer)


def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    px.line(utils.to_numpy(tensor), labels={"x": xaxis, "y": yaxis}, **kwargs).show(
        renderer
    )


def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    x = utils.to_numpy(x)
    y = utils.to_numpy(y)
    px.scatter(
        y=y, x=x, labels={"x": xaxis, "y": yaxis, "color": caxis}, **kwargs
    ).show(renderer)


In [8]:
device = utils.get_device()
device

device(type='cuda')

# Load model

HookedTransformer is a somewhat adapted GPT-2 architecture, but is computationally identical. The most significant changes are to the internal structure of the attention heads:

- The weights (W_K, W_Q, W_V) mapping the residual stream to queries, keys and values are 3 separate matrices, rather than big concatenated one.
- The weight matrices (W_K, W_Q, W_V, W_O) and activations (keys, queries, values, z (values mixed by attention pattern)) have separate head_index and d_head axes, rather than flattening them into one big axis.
  - The activations all have shape `[batch, position, head_index, d_head]`
  - W_K, W_Q, W_V have shape `[head_index, d_model, d_head]` and W_O has shape `[head_index, d_head, d_model]`


In [9]:
def assert_hf_and_tl_model_are_close(
    hf_model,
    tl_model,
    tokenizer,
    prompt="12345 x 54321 = ",
    atol=1e-5,
):
    prompt_toks = tokenizer(prompt, return_tensors="pt").input_ids

    hf_logits = hf_model(prompt_toks.to(hf_model.device)).logits
    tl_logits = tl_model(prompt_toks).to(hf_logits)

    assert torch.allclose(
        torch.softmax(hf_logits, dim=-1), torch.softmax(tl_logits, dim=-1), atol=atol
    )


In [10]:
# NBVAL_IGNORE_OUTPUT
# sliding window warning is false: https://github.com/huggingface/transformers/pull/36316

model_path = "Qwen/Qwen2.5-0.5B"

tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)

hf_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map=device,
    trust_remote_code=True,
).eval()

tl_model = HookedTransformer.from_pretrained_no_processing(
    model_path,
    device=device,
    dtype=torch.float32,
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
).to(device)

assert_hf_and_tl_model_are_close(hf_model, tl_model, tokenizer)


Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer
Moving model to device:  cuda


In [11]:
# tokenizer = AutoTokenizer.from_pretrained(model_path)

# hf_model = AutoModelForCausalLM.from_pretrained(
#     model_path,
#     device_map=device,
# ).eval()

# tl_model = HookedTransformer.from_pretrained_no_processing(
#     model_path,
#     device=device,
#     dtype=torch.float32,
# ).to(device)

# assert_hf_and_tl_model_are_close(hf_model, tl_model, tokenizer)


In [12]:
example_problem = "23456 x 65432 = "
loss = tl_model(example_problem, return_type="loss")
print("Model loss:", loss)


Model loss: tensor(2.4532, device='cuda:0')


# Multi-digit multiplication

The next step is to verify that the model can _actually_ do the task!


In [30]:
# Generate multiplication problems with varying digit counts
problems = []
answers = []
num_digits = 5

# Create numbers with num_digits digits
num1 = np.random.randint(10 ** (num_digits - 1), 10**num_digits)
num2 = np.random.randint(10 ** (num_digits - 1), 10**num_digits)
problems.append(f"{num1} x {num2} =")
answers.append(" " + str(num1 * num2))

# Display the problems
for problem, answer in zip(problems, answers):
    print(problem + answer)


57916 x 43005 = 2490677580


In [31]:
utils.test_prompt(problems[0], answers[0], tl_model, prepend_bos=True)


Tokenized prompt: ['<|endoftext|>', '5', '7', '9', '1', '6', ' x', ' ', '4', '3', '0', '0', '5', ' =']
Tokenized answer: [' ', '2', '4', '9', '0', '6', '7', '7', '5', '8', '0']


Performance on answer token:
Rank: 0        Logit: 21.24 Prob: 86.60% Token: | |

Top 0th token. Logit: 21.24 Prob: 86.60% Token: | |
Top 1th token. Logit: 18.25 Prob:  4.33% Token: | ?
|
Top 2th token. Logit: 18.09 Prob:  3.71% Token: | ?|
Top 3th token. Logit: 17.20 Prob:  1.52% Token: | ?

|
Top 4th token. Logit: 15.79 Prob:  0.37% Token: | x|
Top 5th token. Logit: 15.42 Prob:  0.26% Token: | (|
Top 6th token. Logit: 14.66 Prob:  0.12% Token: | __|
Top 7th token. Logit: 14.64 Prob:  0.12% Token: |<|endoftext|>|
Top 8th token. Logit: 14.49 Prob:  0.10% Token: | $|
Top 9th token. Logit: 14.46 Prob:  0.10% Token: |?
|


Performance on answer token:
Rank: 0        Logit: 22.79 Prob: 84.81% Token: |2|

Top 0th token. Logit: 22.79 Prob: 84.81% Token: |2|
Top 1th token. Logit: 19.81 Prob:  4.28% Token: |3|
Top 2th token. Logit: 19.52 Prob:  3.22% Token: |1|
Top 3th token. Logit: 19.51 Prob:  3.18% Token: |5|
Top 4th token. Logit: 18.78 Prob:  1.54% Token: |4|
Top 5th token. Logit: 18.27 Prob:  0.92% Token: |7|
Top 6th token. Logit: 17.89 Prob:  0.63% Token: |9|
Top 7th token. Logit: 17.79 Prob:  0.57% Token: |6|
Top 8th token. Logit: 17.53 Prob:  0.44% Token: |8|
Top 9th token. Logit: 17.37 Prob:  0.37% Token: |0|


...

Performance on answer token:
Rank: 11       Logit: 15.39 Prob:  1.11% Token: |8|

Top 0th token. Logit: 19.40 Prob: 60.75% Token: |0|
Top 1th token. Logit: 17.99 Prob: 14.78% Token: |
|
Top 2th token. Logit: 17.08 Prob:  5.96% Token: |5|
Top 3th token. Logit: 15.86 Prob:  1.75% Token: |1|
Top 4th token. Logit: 15.85 Prob:  1.74% Token: |

|
Top 5th token. Logit: 15.80 Prob:  1.66% Token: |6|
Top 6th token. Logit: 15.73 Prob:  1.55% Token: |2|
Top 7th token. Logit: 15.64 Prob:  1.42% Token: |3|
Top 8th token. Logit: 15.56 Prob:  1.31% Token: |7|
Top 9th token. Logit: 15.56 Prob:  1.30% Token: |4|


Performance on answer token:
Rank: 1        Logit: 17.56 Prob: 16.40% Token: |0|

Top 0th token. Logit: 17.91 Prob: 23.26% Token: |
|
Top 1th token. Logit: 17.56 Prob: 16.40% Token: |0|
Top 2th token. Logit: 17.39 Prob: 13.86% Token: |5|
Top 3th token. Logit: 16.44 Prob:  5.33% Token: |7|
Top 4th token. Logit: 16.31 Prob:  4.70% Token: |1|
Top 5th token. Logit: 16.28 Prob:  4.55% Token: |2|
Top 6th token. Logit: 16.16 Prob:  4.01% Token: |6|
Top 7th token. Logit: 16.11 Prob:  3.82% Token: |4|
Top 8th token. Logit: 16.06 Prob:  3.66% Token: |3|
Top 9th token. Logit: 16.01 Prob:  3.47% Token: |8|


Ranks of the answer tokens: [(' ', 0), ('2', 0), ('4', 0), ('9', 0), ('0', 8), ('6', 5), ('7', 4), ('7', 6), ('5', 
0), ('8', 11), ('0', 1)]

We now want to find a reference prompt to run the model on. Even though our ultimate goal is to reverse engineer how this behaviour is done in general, often the best way to start out in mechanistic interpretability is by zooming in on a concrete example and understanding it in detail, and only _then_ zooming out and verifying that our analysis generalises.

We'll run the model on 8 instances of this task. To make our lives easier, we'll carefully choose prompts with single token names and the corresponding names in the same token positions.

<details> <summary>(*) <b>Aside on tokenization</b></summary>

We want models that can take in arbitrary text, but models need to have a fixed vocabulary. So the solution is to define a vocabulary of **tokens** and to deterministically break up arbitrary text into tokens. Tokens are, essentially, subwords, and are determined by finding the most frequent substrings - this means that tokens vary a lot in length and frequency!

Tokens are a _massive_ headache and are one of the most annoying things about reverse engineering language models... Different names will be different numbers of tokens, different prompts will have the relevant tokens at different positions, different prompts will have different total numbers of tokens, etc. Language models often devote significant amounts of parameters in early layers to convert inputs from tokens to a more sensible internal format (and do the reverse in later layers). You really, really want to avoid needing to think about tokenization wherever possible when doing exploratory analysis (though, of course, it's relevant later when trying to flesh out your analysis and make it rigorous!). HookedTransformer comes with several helper methods to deal with tokens: `to_tokens, to_string, to_str_tokens, to_single_token, get_token_position`

**Exercise:** I recommend using `model.to_str_tokens` to explore how the model tokenizes different strings. In particular, try adding or removing spaces at the start, or changing capitalization - these change tokenization!</details>


In [38]:
prompt_format = []
insert = []
num_digits = 5

for _ in range(3):
    num1 = np.random.randint(10 ** (num_digits - 1), 10**num_digits)
    num2 = np.random.randint(10 ** (num_digits - 1), 10**num_digits)
    prompt_format.append(f"{num1} x {num2} =")
    prompt_format.append(f"{num2} x {num1} =")
    answer = str(num1 * num2)
    corr_answer = answer[:-1] + str(np.random.randint(1, 10))
    for _ in range(2):
        insert.append((f" {answer}", f" {corr_answer}"))

print(prompt_format)
print(insert)

['74722 x 32649 =', '32649 x 74722 =', '92456 x 95971 =', '95971 x 92456 =', '70163 x 62207 =', '62207 x 70163 =']
[(' 2439598578', ' 2439598577'), (' 2439598578', ' 2439598577'), (' 8873094776', ' 8873094774'), (' 8873094776', ' 8873094774'), (' 4364629741', ' 4364629744'), (' 4364629741', ' 4364629744')]


In [53]:
prompts = []
answers = []  # List of answers, in the format (correct, incorrect)
answer_tokens = []  # List of the token (ie an integer) corresponding to each answer, in the format (correct_token, incorrect_token)

for i in range(len(prompt_format)):
    prompts.append(prompt_format[i])
    answers.append((insert[i][0], insert[i][1]))
    answer_tokens.append(
        (
            tl_model.to_tokens(answers[-1][0]).tolist(),
            tl_model.to_tokens(answers[-1][1]).tolist(),
        )
    )
answer_tokens = torch.tensor(answer_tokens).to(device)

print(prompts)
print(answers)
print(answer_tokens.shape)

['74722 x 32649 =', '32649 x 74722 =', '92456 x 95971 =', '95971 x 92456 =', '70163 x 62207 =', '62207 x 70163 =']
[(' 2439598578', ' 2439598577'), (' 2439598578', ' 2439598577'), (' 8873094776', ' 8873094774'), (' 8873094776', ' 8873094774'), (' 4364629741', ' 4364629744'), (' 4364629741', ' 4364629744')]
torch.Size([6, 2, 1, 11])


**Gotcha**: It's important that all of your prompts have the same number of tokens. If they're different lengths, then the position of the "final" logit where you can check logit difference will differ between prompts, and this will break the below code. The easiest solution is just to choose your prompts carefully to have the same number of tokens (you can eg add filler words like The, or newlines to start).

There's a range of other ways of solving this, eg you can index more intelligently to get the final logit. A better way is to just use left padding by setting `model.tokenizer.padding_side = 'left'` before tokenizing the inputs and running the model; this way, you can use something like `logits[:, -1, :]` to easily access the final token outputs without complicated indexing. TransformerLens checks the value of `padding_side` of the tokenizer internally, and if the flag is set to be `'left'`, it adjusts the calculation of absolute position embedding and causal masking accordingly.

In this demo, though, we stick to using the prompts of the same number of tokens because we want to show some visualisations aggregated along the batch dimension later in the demo.


In [54]:
for prompt in prompts:
    str_tokens = tl_model.to_str_tokens(prompt)
    print("Prompt length:", len(str_tokens))
    print("Prompt as tokens:", str_tokens)


Prompt length: 13
Prompt as tokens: ['7', '4', '7', '2', '2', ' x', ' ', '3', '2', '6', '4', '9', ' =']
Prompt length: 13
Prompt as tokens: ['3', '2', '6', '4', '9', ' x', ' ', '7', '4', '7', '2', '2', ' =']
Prompt length: 13
Prompt as tokens: ['9', '2', '4', '5', '6', ' x', ' ', '9', '5', '9', '7', '1', ' =']
Prompt length: 13
Prompt as tokens: ['9', '5', '9', '7', '1', ' x', ' ', '9', '2', '4', '5', '6', ' =']
Prompt length: 13
Prompt as tokens: ['7', '0', '1', '6', '3', ' x', ' ', '6', '2', '2', '0', '7', ' =']
Prompt length: 13
Prompt as tokens: ['6', '2', '2', '0', '7', ' x', ' ', '7', '0', '1', '6', '3', ' =']


In [55]:
tokens = tl_model.to_tokens(prompts, prepend_bos=True)

# Run the model and cache all activations
original_logits, cache = tl_model.run_with_cache(tokens)


We'll later be evaluating how model performance differs upon performing various interventions, so it's useful to have a metric to measure model performance. Our metric here will be the **logit difference**, the difference in logit between the right and wrong product (eg, `logit(10)-logit(11)`).


In [57]:
def logits_to_ave_logit_diff(logits, answer_tokens, per_prompt=False):
    # Only the final logits are relevant for the answer
    final_logits = logits[:, -1, :]
    print(final_logits.shape, answer_tokens.shape)
    answer_logits = final_logits.gather(dim=-1, index=answer_tokens)
    answer_logit_diff = answer_logits[:, 0] - answer_logits[:, 1]
    if per_prompt:
        return answer_logit_diff
    else:
        return answer_logit_diff.mean()


print(
    "Per prompt logit difference:",
    logits_to_ave_logit_diff(original_logits, answer_tokens, per_prompt=True)
    .detach()
    .cpu()
    .round(decimals=3),
)
original_average_logit_diff = logits_to_ave_logit_diff(original_logits, answer_tokens)
print(
    "Average logit difference:",
    round(logits_to_ave_logit_diff(original_logits, answer_tokens).item(), 3),
)


torch.Size([6, 151936]) torch.Size([6, 2, 1, 11])


RuntimeError: Index tensor must have the same number of dimensions as input tensor

# Cache layer activations

Look at all model activations


## Parameter Names

Here is a list of the parameters and shapes in the model. By convention, all weight matrices multiply on the right (ie `new_activation = old_activation @ weights + bias`).

Reminder of the key hyper-params:

- `n_layers`: 12. The number of transformer blocks in the model (a block contains an attention layer and an MLP layer)
- `n_heads`: 14. The number of attention heads per attention layer
- `d_model`: 896. The residual stream width.
- `d_head`: 64. The internal dimension of an attention head activation.
- `d_mlp`: 4864. The internal dimension of the MLP layers (ie the number of neurons).
- `d_vocab`: 151936. The number of tokens in the vocabulary.
- `n_ctx`: 128000. The maximum number of tokens in an input prompt.


In [ ]:
for name, param in tl_model.named_parameters():
    if name.startswith("blocks.0."):
        print(name, param.shape)


In [ ]:
for name, param in tl_model.named_parameters():
    if not name.startswith("blocks"):
        print(name, param.shape)


## Activation + Hook Names

Let's do this by entering in a short prompt and add a hook function to each activations to print its name and shape. To avoid spam, let's just add this to activations in the first block or not in a block.

Note 1: Each LayerNorm has a hook for the scale factor (ie the standard deviation of the input activations for each token position & batch element) and for the normalized output (ie the input activation with mean 0 and standard deviation 1, but _before_ applying scaling or translating with learned weights). LayerNorm is applied every time a layer reads from the residual stream: `ln1` is the LayerNorm before the attention layer in a block, `ln2` the one before the MLP layer, and `ln_final` is the LayerNorm before the unembed.

Note 2: _Every_ activation apart from the attention pattern and attention scores has shape beginning with `[batch, position]`. The attention pattern and scores have shape `[batch, head_index, dest_position, source_position]` (the numbers are the same, unless we're using caching).


In [ ]:
example_problem = "45678 x 87654 = "
print("Num tokens:", len(tl_model.to_tokens(example_problem)[0]))


def print_name_shape_hook_function(activation, hook):
    print(hook.name, activation.shape)


def not_in_late_block_filter(name):
    return name.startswith("blocks.0.") or not name.startswith("blocks")


tl_model.run_with_hooks(
    example_problem,
    return_type=None,
    fwd_hooks=[(not_in_late_block_filter, print_name_shape_hook_function)],
)

## Folding LayerNorm

LayerNorm is a normalization technique used by transformers, analogous to BatchNorm but more friendly to massive parallelisation. No one _really_ knows why it works, but it seems to improve model numerical stability. Unlike BatchNorm, LayerNorm actually changes the functional form of the model, which makes it a massive pain for interpretability!

Folding LayerNorm is a technique to make it lower overhead to deal with, and the flags `center_writing_weights` and `fold_ln` in `HookedTransformer.from_pretrained` apply this automatically (they default to True). These simplify the internal structure without changing the weights.

Intuitively, LayerNorm acts on each residual stream vector (ie for each batch element and token position) independently, sets their mean to 0 (centering) and standard deviation to 1 (normalizing) (_across_ the residual stream dimension - very weird!), and then applies a learned elementwise scaling and translation to each vector.

Mathematically, centering is a linear map, normalizing is _not_ a linear map, and scaling and translation are linear maps.

- **Centering:** LayerNorm is applied every time a layer reads from the residual stream, so the mean of any residual stream vector can never matter - `center_writing_weights` set every weight matrix writing to the residual to have zero mean.
- **Normalizing:** Normalizing is not a linear map, and cannot be factored out. The `hook_scale` hook point lets you access and control for this.
- **Scaling and Translation:** Scaling and translation are linear maps, and are always followed by another linear map. The composition of two linear maps is another linear map, so we can _fold_ the scaling and translation weights into the weights of the subsequent layer, and simplify things without changing the underlying computation.

[See the docs for more details](https://github.com/TransformerLensOrg/TransformerLens/blob/main/further_comments.md#what-is-layernorm-folding-fold_ln)

A fun consequence of LayerNorm folding is that it creates a bias across the unembed, a `d_vocab` length vector that is added to the output logits - GPT-2 is not trained with this, but it _is_ trained with a final LayerNorm that contains a bias.

Turns out, this LayerNorm bias learns structure of the data that we can only see after folding! In particular, it essentially learns **unigram statistics** - rare tokens get suppressed, common tokens get boosted, by pretty dramatic degrees! Let's list the top and bottom 20 - at the top we see common punctuation and words like " the" and " and", at the bottom we see weird-ass tokens like " RandomRedditor":


In [ ]:
unembed_bias = tl_model.unembed.b_U
bias_values, bias_indices = unembed_bias.sort(descending=True)
top_k = 20
print(f"Top {top_k} values")
for i in range(top_k):
    print(f"{bias_values[i].item():.2f} {repr(tl_model.to_string(bias_indices[i]))}")

print("...")
print(f"Bottom {top_k} values")
for i in range(top_k, 0, -1):
    print(f"{bias_values[-i].item():.2f} {repr(tl_model.to_string(bias_indices[-i]))}")

In [ ]:
# times_bias = tl_model.unembed.b_U[tl_model.to_single_token(" times")]
# mult_bias = tl_model.unembed.b_U[tl_model.to_single_token(" multiplied")]

# print(f"'times' bias: {times_bias.item():.4f}")
# print(f"'multiplied' bias: {mult_bias.item():.4f}")
# print(f"Prob ratio bias: {torch.exp(times_bias - mult_bias).item():.4f}x")


## e.g. cache layer 0 attention act

Get only layer 0 attention patterns to save on VRAM by stopping the forward pass at layer 1.


In [ ]:
example_problem = "56789 x 98765 = "
model_tokens = tl_model.to_tokens(example_problem)
print(model_tokens.device)

attn_hook_name = "blocks.0.attn.hook_pattern"
attn_layer = 0
_, attn_cache = tl_model.run_with_cache(
    model_tokens,
    remove_batch_dim=True,
    stop_at_layer=attn_layer + 1,
    names_filter=[attn_hook_name],
)

print(type(attn_cache))
attn = attn_cache[attn_hook_name]
print(attn.shape)
gpt2_str_tokens = tl_model.to_str_tokens(example_problem)

In [ ]:
print("Layer 0 Head Attention Patterns:")
cv.attention.attention_patterns(tokens=model_tokens, attention=attn)

# Hooks: intervene on acts

One of the great things about interpreting neural networks is that we have _full control_ over our system. From a computational perspective, we know exactly what operations are going on inside (even if we don't know what they mean!). And we can make precise, surgical edits and see how the model's behaviour and other internals change. This is an extremely powerful tool, because it can let us eg set up careful counterfactuals and causal intervention to easily understand model behaviour.

Accordingly, being able to do this is a pretty core operation, and this is one of the main things TransformerLens supports! The key feature here is **hook points**. Every activation inside the transformer is surrounded by a hook point, which allows us to edit or intervene on it.

We do this by adding a **hook function** to that activation. The hook function maps `current_activation_value, hook_point` to `new_activation_value`. As the model is run, it computes that activation as normal, and then the hook function is applied to compute a replacement, and that is substituted in for the activation. The hook function can be an arbitrary Python function, so long as it returns a tensor of the correct shape.

<details><summary>Relationship to PyTorch hooks</summary>

[PyTorch hooks](https://blog.paperspace.com/pytorch-hooks-gradient-clipping-debugging/) are a great and underrated, yet incredibly janky, feature. They can act on a layer, and edit the input or output of that layer, or the gradient when applying autodiff. The key difference is that **Hook points** act on _activations_ not layers. This means that you can intervene within a layer on each activation, and don't need to care about the precise layer structure of the transformer. And it's immediately clear exactly how the hook's effect is applied. This adjustment was shamelessly inspired by [Garcon's use of ProbePoints](https://transformer-circuits.pub/2021/garcon/index.html).

They also come with a range of other quality of life improvements, like the model having a `model.reset_hooks()` method to remove all hooks, or helper methods to temporarily add hooks for a single forward pass - it is _incredibly_ easy to shoot yourself in the foot with standard PyTorch hooks!

</details>

As a basic example, let's [ablate](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=fh-HJyz1CgUVrXuoiban6bYx) head 1 in layer 0 on the text above.

We define a `head_ablation_hook` function. This takes the value tensor for attention layer 0, and sets the component with `head_index==1` to zero and returns it (Note - we return by convention, but since we're editing the activation in-place, we don't strictly _need_ to).

We then use the `run_with_hooks` helper function to run the model and _temporarily_ add in the hook for just this run. We enter in the hook as a tuple of the activation name (also the hook point name - found with `utils.get_act_name`) and the hook function.

**Gotcha:** Hooks are global state - they're added in as part of the model, and stay there until removed. `run_with_hooks` tries to create an abstraction where these are local state, by removing all hooks at the end of the function. But you can easily shoot yourself in the foot if there's, eg, an error in one of your hooks so the function never finishes. If you start getting bugs, try `model.reset_hooks()` to clean things up. Further, if you _do_ add hooks of your own that you want to keep, which you can do with `add_perma_hook` on the relevant HookPoint


In [ ]:
layer_to_ablate = 0
head_index_to_ablate = 1


def head_ablation_hook(
    value: Float[torch.Tensor, "batch pos head_index d_head"], hook: HookPoint
) -> Float[torch.Tensor, "batch pos head_index d_head"]:
    print(f"Shape of the value tensor: {value.shape}")
    value[:, :, head_index_to_ablate, :] = 0.0
    return value


original_loss = tl_model(model_tokens, return_type="loss")
ablated_loss = tl_model.run_with_hooks(
    model_tokens,
    return_type="loss",
    fwd_hooks=[(utils.get_act_name("v", layer_to_ablate), head_ablation_hook)],
)
print(f"Original Loss: {original_loss.item():.3f}")
print(f"Ablated Loss: {ablated_loss.item():.3f}")


## Act Patch on Multiplication Task

For a somewhat more involved example, let's use hooks to apply **[activation patching](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=qeWBvs-R-taFfcCq-S_hgMqx)** on the task of multiplication.

**[Activation patching](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=qeWBvs-R-taFfcCq-S_hgMqx)** is a technique from [Kevin Meng and David Bau's excellent ROME paper](https://rome.baulab.info/). The goal is to identify which model activations are important for completing a task. We do this by setting up a **clean prompt** and a **corrupted prompt** and a **metric** for performance on the task. We then pick a specific model activation, run the model on the corrupted prompt, but then _intervene_ on that activation and patch in its value when run on the clean prompt. We then apply the metric, and see how much this patch has recovered the clean performance.
(See [a more detailed demonstration of activation patching here](https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Exploratory_Analysis_Demo.ipynb))


In [ ]:
clean_prompt = "56789 x 98765 = "
corrupted_prompt = "56789 x 98760 = "

clean_tokens = tl_model.to_tokens(clean_prompt)
corrupted_tokens = tl_model.to_tokens(corrupted_prompt)


def logits_to_logit_diff(
    logits, correct_answer=" 5608765585", incorrect_answer=" 5608481640"
):
    # model.to_single_token maps a string value of a single token to the token index for that token
    # If the string is not a single token, it raises an error.

    # correct_index = tl_model.to_single_token(correct_answer)
    # incorrect_index = tl_model.to_single_token(incorrect_answer)
    # return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

    correct_idxs = tl_model.to_tokens(correct_answer)
    incorrect_idxs = tl_model.to_tokens(incorrect_answer)
    return (logits[0, -1, correct_idxs] - logits[0, -1, incorrect_idxs]).mean()


# We run on the clean prompt with the cache so we store activations to patch in later.
clean_logits, clean_cache = tl_model.run_with_cache(clean_tokens)
clean_logit_diff = logits_to_logit_diff(clean_logits)
print(f"Clean logit difference: {clean_logit_diff.item():.3f}")

# We don't need to cache on the corrupted prompt.
corrupted_logits = tl_model(corrupted_tokens)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)
print(f"Corrupted logit difference: {corrupted_logit_diff.item():.3f}")


We now setup the hook function to do activation patching. Here, we'll patch in the [residual stream](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=DHp9vZ0h9lA9OCrzG2Y3rrzH) at the start of a specific layer and at a specific position. This will let us see how much the model is using the residual stream at that layer and position to represent the key information for the task.

We want to iterate over all layers and positions, so we write the hook to take in an position parameter. Hook functions must have the input signature (activation, hook), but we can use `functools.partial` to set the position parameter before passing it to `run_with_hooks`


In [ ]:
# We define a residual stream patching hook
# We choose to act on the residual stream at the start of the layer, so we call it resid_pre
# The type annotations are a guide to the reader and are not necessary
def residual_stream_patching_hook(
    resid_pre: Float[torch.Tensor, "batch pos d_model"], hook: HookPoint, position: int
) -> Float[torch.Tensor, "batch pos d_model"]:
    # Each HookPoint has a name attribute giving the name of the hook.
    clean_resid_pre = clean_cache[hook.name]
    resid_pre[:, position, :] = clean_resid_pre[:, position, :]
    return resid_pre


# We make a tensor to store the results for each patching run. We put it on the model's device to avoid needing to move things between the GPU and CPU, which can be slow.
num_positions = len(clean_tokens[0])
ioi_patching_result = torch.zeros(
    (tl_model.cfg.n_layers, num_positions), device=tl_model.cfg.device
)

for layer in tqdm.tqdm(range(tl_model.cfg.n_layers)):
    for position in range(num_positions):
        # Use functools.partial to create a temporary hook function with the position fixed
        temp_hook_fn = partial(residual_stream_patching_hook, position=position)
        # Run the model with the patching hook
        patched_logits = tl_model.run_with_hooks(
            corrupted_tokens,
            fwd_hooks=[(utils.get_act_name("resid_pre", layer), temp_hook_fn)],
        )
        # Calculate the logit difference
        patched_logit_diff = logits_to_logit_diff(patched_logits).detach()
        # Store the result, normalizing by the clean and corrupted logit difference so it's between 0 and 1 (ish)
        ioi_patching_result[layer, position] = (
            patched_logit_diff - corrupted_logit_diff
        ) / (clean_logit_diff - corrupted_logit_diff)


We can now visualize the results, and see that this computation is extremely localised within the model. Initially, the last digit of the second term token is all that matters (naturally, as it's the only different token), and all relevant information remains here until heads in layer 14 and 15 move this to the final token where it's used to predict the answer.
(Note - the heads are in layer 14 and 15, not 15 and 16, because we patched in the residual stream at the _start_ of each layer)


In [ ]:
# Add the index to the end of the label, because plotly doesn't like duplicate labels
token_labels = [
    f"{token}_{index}"
    for index, token in enumerate(tl_model.to_str_tokens(clean_tokens))
]
imshow(
    ioi_patching_result,
    x=token_labels,
    xaxis="Position",
    yaxis="Layer",
    title="Normalized Logit Difference After Patching Residual Stream on the IOI Task",
)


# Access Act

Hooks can also be used to just **access** an activation - to run some function using that activation value, _without_ changing the activation value. This can be achieved by just having the hook return nothing, and not editing the activation in place.

This is useful for eg extracting activations for a specific task, or for doing some long-running calculation across many inputs, eg finding the text that most activates a specific neuron. (Note - everything this can do _could_ be done with `run_with_cache` and post-processing, but this workflow can be more intuitive and memory efficient.)

To demonstrate this, let's look for **[induction heads](https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html)**.

Induction circuits are a very important circuit in generative language models, which are used to detect and continue repeated subsequences. They consist of two heads in separate layers that compose together, a **previous token head** which always attends to the previous token, and an **induction head** which attends to the token _after_ an earlier copy of the current token.

To see why this is important, let's say that the model is trying to predict the next token in a news article about Michael Jordan. The token " Michael", in general, could be followed by many surnames. But an induction head will look from that occurrence of " Michael" to the token after previous occurrences of " Michael", ie " Jordan" and can confidently predict that that will come next.

An interesting fact about induction heads is that they generalise to arbitrary sequences of repeated tokens. We can see this by generating sequences of 50 random tokens, repeated twice, and plotting the average loss at predicting the next token, by position. We see that the model goes from terrible to very good at the halfway point.


In [ ]:
batch_size = 10
seq_len = 50
size = (batch_size, seq_len)
input_tensor = torch.randint(1000, 10000, size)

random_tokens = input_tensor.to(tl_model.cfg.device)
repeated_tokens = einops.repeat(random_tokens, "batch seq_len -> batch (2 seq_len)")
repeated_logits = tl_model(repeated_tokens)
correct_log_probs = tl_model.loss_fn(repeated_logits, repeated_tokens, per_token=True)
loss_by_position = einops.reduce(
    correct_log_probs, "batch position -> position", "mean"
)
line(
    loss_by_position,
    xaxis="Position",
    yaxis="Loss",
    title="Loss by position on random repeated tokens",
)


The induction heads will be attending from the second occurrence of each token to the token _after_ its first occurrence, ie the token `50-1==49` places back. So by looking at the average attention paid 49 tokens back, we can identify induction heads! Let's define a hook to do this!

<details><summary>Technical details</summary>

- We attach the hook to the attention pattern activation. There's one big pattern activation per layer, stacked across all heads, so we need to do some tensor manipulation to get a per-head score.
- Hook functions can access global state, so we make a big tensor to store the induction head score for each head, and then we just add the score for each head to the appropriate position in the tensor.
- To get a single hook function that works for each layer, we use the `hook.layer()` method to get the layer index (internally this is just inferred from the hook names).
- As we want to add this to _every_ activation pattern hook point, rather than giving the string for an activation name, this time we give a **name filter**. This is a Boolean function on hook point names, and it adds the hook function to every hook point where the function evaluates as true. \* `run_with_hooks` allows us to enter a list of (act_name, hook_function) pairs to all be added at once, so we could also have done this by inputting a list with a hook for each layer.
</details>


In [ ]:
# We make a tensor to store the induction score for each head. We put it on the model's device to avoid needing to move things between the GPU and CPU, which can be slow.
induction_score_store = torch.zeros(
    (tl_model.cfg.n_layers, tl_model.cfg.n_heads), device=tl_model.cfg.device
)


def induction_score_hook(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    # We take the diagonal of attention paid from each destination position to source positions seq_len-1 tokens back
    # (This only has entries for tokens with index>=seq_len)
    induction_stripe = pattern.diagonal(dim1=-2, dim2=-1, offset=1 - seq_len)
    # Get an average score per head
    induction_score = einops.reduce(
        induction_stripe, "batch head_index position -> head_index", "mean"
    )
    # Store the result.
    induction_score_store[hook.layer(), :] = induction_score


# We make a boolean filter on activation names, that's true only on attention pattern names.
def pattern_hook_names_filter(name):
    return name.endswith("pattern")


tl_model.run_with_hooks(
    repeated_tokens,
    return_type=None,  # For efficiency, we don't need to calculate the logits
    fwd_hooks=[(pattern_hook_names_filter, induction_score_hook)],
)

imshow(
    induction_score_store, xaxis="Head", yaxis="Layer", title="Induction Score by Head"
)


Head 3 in Layer 16 scores extremely highly on this score, and we can feed in a shorter repeated random sequence, visualize the attention pattern for it and see this directly - including the "induction stripe" at `seq_len-1` tokens back.

This time we put in a hook on the attention pattern activation to visualize the pattern of the relevant head.


In [ ]:
induction_head_layer = 16
induction_head_index = 3
size = (1, 20)
input_tensor = torch.randint(1000, 10000, size)

single_random_sequence = input_tensor.to(tl_model.cfg.device)
repeated_random_sequence = einops.repeat(
    single_random_sequence, "batch seq_len -> batch (2 seq_len)"
)


def visualize_pattern_hook(
    pattern: Float[torch.Tensor, "batch head_index dest_pos source_pos"],
    hook: HookPoint,
):
    display(
        cv.attention.attention_patterns(
            tokens=tl_model.to_str_tokens(repeated_random_sequence),
            attention=pattern[0, induction_head_index, :, :][
                None, :, :
            ],  # Add a dummy axis, as CircuitsVis expects 3D patterns.
        )
    )


tl_model.run_with_hooks(
    repeated_random_sequence,
    return_type=None,
    fwd_hooks=[
        (utils.get_act_name("pattern", induction_head_layer), visualize_pattern_hook)
    ],
)
